In [3]:
# Importation de NumPy
# Cette bibliothèque permet de faire des calculs mathématiques rapides
# et de manipuler des tableaux numériques.
import numpy as np

# Importation de Pandas
# Pandas permet de créer et manipuler des DataFrames
# pour organiser les données du dataset.
import pandas as pd

# Importation de Matplotlib
# Cette bibliothèque sert à créer les graphes et visualisations.
import matplotlib.pyplot as plt

In [4]:
# ============================================================
# on commence avec l'implementation suivant ID3 puis on passera au c4.5 et finalement CART 
# ============================================================

# Pour bien comprendre ID3, on commence avec un petit dataset catégoriel.
# ID3 fonctionne mieux avec des variables catégorielles.

data_id3 = pd.DataFrame({
    "meteo": ["soleil", "soleil", "nuage", "pluie", "pluie", "pluie", "nuage", "soleil", "soleil", "pluie"],
    "temperature": ["chaud", "chaud", "chaud", "doux", "froid", "froid", "froid", "doux", "froid", "doux"],
    "vent": ["faible", "fort", "faible", "faible", "faible", "fort", "fort", "faible", "faible", "faible"],
    "jouer": ["non", "non", "oui", "oui", "oui", "non", "oui", "non", "oui", "oui"]
})

data_id3

,meteo,temperature,vent,jouer
0,soleil,chaud,faible,non
1,soleil,chaud,fort,non
2,nuage,chaud,faible,oui
3,pluie,doux,faible,oui
4,pluie,froid,faible,oui
5,pluie,froid,fort,non
6,nuage,froid,fort,oui
7,soleil,doux,faible,non
8,soleil,froid,faible,oui
9,pluie,doux,faible,oui


In [5]:
def entropie(y):
    """
    Calcule l'entropie d'une variable cible.
    
    y : colonne contenant les classes
    
    L'entropie mesure le désordre.
    Si toutes les classes sont identiques, l'entropie vaut 0.
    """
    
    # Récupérer les classes différentes et leur nombre d'apparitions
    classes, counts = np.unique(y, return_counts=True)
    
    # Calculer les probabilités de chaque classe
    probabilites = counts / counts.sum()
    
    # Calcul de l'entropie : - somme(p * log2(p))
    entropy = -np.sum(probabilites * np.log2(probabilites))
    
    return entropy

In [6]:
print("Entropie de la cible jouer :", entropie(data_id3["jouer"]))

Entropie de la cible jouer : 0.9709505944546686


In [7]:
def gain_information(data, feature, target):
    """
    Calcule le gain d'information d'une variable.
    
    data : dataset complet
    feature : variable explicative à tester
    target : variable cible
    
    Gain = entropie parent - entropie pondérée des sous-groupes
    """
    
    # Entropie avant la séparation
    entropie_parent = entropie(data[target])
    
    # Valeurs possibles de la variable
    valeurs = data[feature].unique()
    
    # Initialisation de l'entropie après split
    entropie_ponderee = 0
    
    # Pour chaque valeur de la variable
    for valeur in valeurs:
        
        # Sous-dataset correspondant à cette valeur
        sous_data = data[data[feature] == valeur]
        
        # Poids du sous-groupe
        poids = len(sous_data) / len(data)
        
        # Entropie du sous-groupe
        entropie_sous_groupe = entropie(sous_data[target])
        
        # Ajout pondéré
        entropie_ponderee += poids * entropie_sous_groupe
    
    # Calcul du gain d'information
    gain = entropie_parent - entropie_ponderee
    
    return gain

In [8]:
features = ["meteo", "temperature", "vent"]

for feature in features:
    print(feature, "=> Gain =", gain_information(data_id3, feature, "jouer"))

meteo => Gain = 0.3219280948873623
temperature => Gain = 0.09546184423832171
vent => Gain = 0.0912774462416801


In [9]:
def meilleure_variable(data, features, target):
    """
    Choisit la variable avec le plus grand gain d'information.
    """
    
    gains = {}
    
    # Calculer le gain pour chaque variable
    for feature in features:
        gains[feature] = gain_information(data, feature, target)
    
    # Choisir la variable qui a le gain maximal
    best_feature = max(gains, key=gains.get)
    
    return best_feature, gains

In [10]:
best, gains = meilleure_variable(data_id3, features, "jouer")

print("Gains :", gains)
print("Meilleure variable :", best)

Gains : {'meteo': np.float64(0.3219280948873623), 'temperature': np.float64(0.09546184423832171), 'vent': np.float64(0.0912774462416801)}
Meilleure variable : meteo


In [11]:
def classe_majoritaire(y):
    """
    Retourne la classe la plus fréquente.
    Utilisée quand on ne peut plus diviser l'arbre.
    """
    
    classes, counts = np.unique(y, return_counts=True)
    
    return classes[np.argmax(counts)]

In [12]:
def construire_id3(data, features, target):
    """
    Construit récursivement un arbre de décision ID3.
    
    Cas d'arrêt :
    1. Toutes les lignes ont la même classe
    2. Il n'y a plus de variables disponibles
    """
    
    # Classes présentes dans le dataset
    classes = data[target].unique()
    
    # Cas 1 : toutes les observations ont la même classe
    if len(classes) == 1:
        return classes[0]
    
    # Cas 2 : plus aucune variable explicative disponible
    if len(features) == 0:
        return classe_majoritaire(data[target])
    
    # Choisir la meilleure variable selon le gain d'information
    best_feature, gains = meilleure_variable(data, features, target)
    
    # Créer un noeud de l'arbre
    arbre = {best_feature: {}}
    
    # Pour chaque valeur possible de la meilleure variable
    for valeur in data[best_feature].unique():
        
        # Créer un sous-dataset
        sous_data = data[data[best_feature] == valeur]
        
        # Supprimer la variable déjà utilisée
        nouvelles_features = [f for f in features if f != best_feature]
        
        # Construire récursivement la branche
        arbre[best_feature][valeur] = construire_id3(
            sous_data,
            nouvelles_features,
            target
        )
    
    return arbre

In [13]:
arbre_id3 = construire_id3(data_id3, features, "jouer")

arbre_id3

{'meteo': {'soleil': {'temperature': {'chaud': 'non',
    'doux': 'non',
    'froid': 'oui'}},
  'nuage': 'oui',
  'pluie': {'vent': {'faible': 'oui', 'fort': 'non'}}}}

In [14]:
def predire_id3(arbre, observation):
    """
    Prédit la classe d'une observation avec l'arbre ID3.
    
    arbre : arbre construit par ID3
    observation : une ligne du dataset ou un dictionnaire
    """
    
    # Si l'arbre est une feuille, on retourne directement la classe
    if not isinstance(arbre, dict):
        return arbre
    
    # Récupérer la variable du noeud courant
    feature = next(iter(arbre))
    
    # Récupérer la valeur de cette variable dans l'observation
    valeur = observation[feature]
    
    # Si cette valeur existe dans l'arbre
    if valeur in arbre[feature]:
        return predire_id3(arbre[feature][valeur], observation)
    
    # Si la valeur n'existe pas, on retourne None
    return None

In [15]:
exemple = {
    "meteo": "nuage",
    "temperature": "chaud",
    "vent": "faible"
}

prediction = predire_id3(arbre_id3, exemple)

print("Prédiction :", prediction)

Prédiction : oui


In [17]:
 # Test sur notre dataset
predictions_id3 = []

for i in range(len(data_id3)):
    observation = data_id3.iloc[i]
    prediction = predire_id3(arbre_id3, observation)
    predictions_id3.append(prediction)

data_id3["prediction_id3"] = predictions_id3

data_id3

,meteo,temperature,vent,jouer,prediction_id3
0,soleil,chaud,faible,non,non
1,soleil,chaud,fort,non,non
2,nuage,chaud,faible,oui,oui
3,pluie,doux,faible,oui,oui
4,pluie,froid,faible,oui,oui
5,pluie,froid,fort,non,non
6,nuage,froid,fort,oui,oui
7,soleil,doux,faible,non,non
8,soleil,froid,faible,oui,oui
9,pluie,doux,faible,oui,oui


Problème de ID3

ID3 utilise :

Information Gain

Mais il a un défaut :

Il peut favoriser une variable avec beaucoup de valeurs différentes.

Exemple :

ID étudiant :
1
2
3
4
5

Chaque ligne devient unique → entropie presque 0 → gain énorme.

Mais cette variable ne généralise pas du tout.

Solution de C4.5 :

C4.5 utilise :

Gain Ratio

Formule :

Gain Ratio = Information Gain / Split Information

Split Information :

Le Split Information mesure combien la variable divise les données.

Formule :

SplitInfo(A) = - Σ pᵢ log₂(pᵢ)

Très similaire à l’entropie.



In [18]:
def split_information(data, feature):
    """
    Calcule le Split Information d'une variable.
    
    Cette mesure pénalise les variables
    qui créent trop de divisions.
    """
    
    # Valeurs possibles de la variable
    valeurs = data[feature].unique()
    
    split_info = 0
    
    # Calcul de la formule :
    # - somme(p * log2(p))
    for valeur in valeurs:
        
        # Sous-groupe correspondant
        sous_data = data[data[feature] == valeur]
        
        # Probabilité du sous-groupe
        p = len(sous_data) / len(data)
        
        split_info -= p * np.log2(p)
    
    return split_info

In [20]:
print(split_information(data_id3, "meteo"))

1.5219280948873621


In [21]:
def gain_ratio(data, feature, target):
    """
    Calcule le Gain Ratio utilisé par C4.5.
    
    Gain Ratio = Information Gain / Split Information
    """
    
    # Calcul du gain d'information
    gain = gain_information(data, feature, target)
    
    # Calcul du split information
    split_info = split_information(data, feature)
    
    # Éviter la division par zéro
    if split_info == 0:
        return 0
    
    # Calcul du Gain Ratio
    ratio = gain / split_info
    
    return ratio

In [22]:
features = ["meteo", "temperature", "vent"]

for feature in features:
    print(feature, "=> Gain Ratio =", gain_ratio(data_id3, feature, "jouer"))

meteo => Gain Ratio = 0.2115264814210478
temperature => Gain Ratio = 0.060766929638204084
vent => Gain Ratio = 0.10357243711623386


In [23]:
def meilleure_variable_c45(data, features, target):
    """
    Choisit la variable ayant le meilleur Gain Ratio.
    """
    
    ratios = {}
    
    # Calcul du Gain Ratio pour chaque variable
    for feature in features:
        ratios[feature] = gain_ratio(data, feature, target)
    
    # Variable avec le meilleur ratio
    best_feature = max(ratios, key=ratios.get)
    
    return best_feature, ratios

In [24]:
best_feature_c45, ratios = meilleure_variable_c45(
    data_id3,
    features,
    "jouer"
)

print("Ratios :", ratios)
print("Meilleure variable :", best_feature_c45)

Ratios : {'meteo': np.float64(0.2115264814210478), 'temperature': np.float64(0.060766929638204084), 'vent': np.float64(0.10357243711623386)}
Meilleure variable : meteo


In [25]:
def construire_c45(data, features, target):
    """
    Construction récursive d'un arbre C4.5.
    """
    
    # Classes présentes
    classes = data[target].unique()
    
    # Cas 1 : toutes les lignes ont la même classe
    if len(classes) == 1:
        return classes[0]
    
    # Cas 2 : plus de variables
    if len(features) == 0:
        return classe_majoritaire(data[target])
    
    # Choisir la meilleure variable selon Gain Ratio
    best_feature, ratios = meilleure_variable_c45(
        data,
        features,
        target
    )
    
    # Création du noeud
    arbre = {best_feature: {}}
    
    # Construire chaque branche
    for valeur in data[best_feature].unique():
        
        # Sous-dataset
        sous_data = data[data[best_feature] == valeur]
        
        # Variables restantes
        nouvelles_features = [
            f for f in features
            if f != best_feature
        ]
        
        # Appel récursif
        arbre[best_feature][valeur] = construire_c45(
            sous_data,
            nouvelles_features,
            target
        )
    
    return arbre

In [26]:
arbre_c45 = construire_c45(
    data_id3,
    features,
    "jouer"
)

arbre_c45

{'meteo': {'soleil': {'temperature': {'chaud': 'non',
    'doux': 'non',
    'froid': 'oui'}},
  'nuage': 'oui',
  'pluie': {'vent': {'faible': 'oui', 'fort': 'non'}}}}

In [27]:
exemple = {
    "meteo": "pluie",
    "temperature": "doux",
    "vent": "faible"
}

prediction_c45 = predire_id3(arbre_c45, exemple)

print("Prédiction C4.5 :", prediction_c45)

Prédiction C4.5 : oui


CART signifie :

Classification And Regression Trees

Il peut faire :

- classification
- régression

Contrairement à ID3 et C4.5 :

 CART construit des arbres binaires uniquement.

Chaque nœud fait une question du type :

superficie <= 120 ?
    Oui
    Non

Critère utilisé : indice de Gini

ID3 → Entropie
C4.5 → Gain Ratio
CART → Gini

Intuition du Gini :

Le Gini mesure l’impureté d’un groupe.

- Cas parfait
[1,1,1,1]

Toutes les classes sont identiques.

Gini = 0

Très bon split.

- Cas mélangé
[1,0,1,0]

Classes mélangées.

Gini élevé

Mauvais split.

Formule du Gini :
Gini = 1 - Σ pᵢ²

où :

pᵢ = probabilité de chaque classe

 Objectif de CART :

Trouver le split qui minimise :

Gini pondéré

Donc :

meilleur split = plus faible impurité

In [28]:
def gini(y):
    """
    Calcule l'indice de Gini.
    
    Gini = 1 - somme(p_i²)
    
    Plus Gini est proche de 0,
    plus le groupe est pur.
    """
    
    # Classes uniques + nombre d'apparitions
    classes, counts = np.unique(y, return_counts=True)
    
    # Probabilités des classes
    probabilites = counts / counts.sum()
    
    # Calcul du Gini
    gini_index = 1 - np.sum(probabilites ** 2)
    
    return gini_index

In [29]:
print(gini(data_id3["jouer"]))

0.48


CART teste des divisions :

variable == valeur

ou

variable <= seuil

In [30]:
def split_dataset(data, feature, valeur):
    """
    Divise le dataset en deux groupes.
    
    Gauche :
        feature == valeur
    
    Droite :
        feature != valeur
    """
    
    gauche = data[data[feature] == valeur]
    
    droite = data[data[feature] != valeur]
    
    return gauche, droite

In [31]:
def gini_pondere(gauche, droite, target):
    """
    Calcule le Gini pondéré après un split.
    """
    
    # Taille totale
    total = len(gauche) + len(droite)
    
    # Poids du groupe gauche
    poids_gauche = len(gauche) / total
    
    # Poids du groupe droite
    poids_droite = len(droite) / total
    
    # Calcul pondéré
    gini_total = (
        poids_gauche * gini(gauche[target])
        + poids_droite * gini(droite[target])
    )
    
    return gini_total

In [32]:
def meilleur_split_cart(data, features, target):
    """
    Recherche le meilleur split CART.
    
    Le meilleur split est celui
    qui minimise le Gini pondéré.
    """
    
    meilleur_feature = None
    meilleure_valeur = None
    
    meilleur_gini = float("inf")
    
    # Tester chaque variable
    for feature in features:
        
        # Tester chaque valeur possible
        for valeur in data[feature].unique():
            
            # Diviser les données
            gauche, droite = split_dataset(
                data,
                feature,
                valeur
            )
            
            # Éviter les groupes vides
            if len(gauche) == 0 or len(droite) == 0:
                continue
            
            # Calcul du Gini pondéré
            score = gini_pondere(
                gauche,
                droite,
                target
            )
            
            # Garder le meilleur split
            if score < meilleur_gini:
                
                meilleur_gini = score
                
                meilleur_feature = feature
                
                meilleure_valeur = valeur
    
    return meilleur_feature, meilleure_valeur, meilleur_gini

In [33]:
feature, valeur, score = meilleur_split_cart(
    data_id3,
    features,
    "jouer"
)

print("Meilleure variable :", feature)
print("Valeur du split :", valeur)
print("Gini :", score)

Meilleure variable : meteo
Valeur du split : soleil
Gini : 0.31666666666666665


In [34]:
def construire_cart(data, features, target, profondeur=0, max_profondeur=3):
    """
    Construction récursive d'un arbre CART.
    """
    
    # Classes présentes
    classes = data[target].unique()
    
    # Cas 1 : groupe pur
    if len(classes) == 1:
        return classes[0]
    
    # Cas 2 : profondeur maximale atteinte
    if profondeur >= max_profondeur:
        return classe_majoritaire(data[target])
    
    # Recherche du meilleur split
    feature, valeur, score = meilleur_split_cart(
        data,
        features,
        target
    )
    
    # Si aucun split trouvé
    if feature is None:
        return classe_majoritaire(data[target])
    
    # Création du noeud
    arbre = {
        feature: {
            "valeur": valeur,
            "gauche": None,
            "droite": None
        }
    }
    
    # Split des données
    gauche, droite = split_dataset(
        data,
        feature,
        valeur
    )
    
    # Construction récursive
    arbre[feature]["gauche"] = construire_cart(
        gauche,
        features,
        target,
        profondeur + 1,
        max_profondeur
    )
    
    arbre[feature]["droite"] = construire_cart(
        droite,
        features,
        target,
        profondeur + 1,
        max_profondeur
    )
    
    return arbre

In [35]:
arbre_cart = construire_cart(
    data_id3,
    features,
    "jouer"
)

arbre_cart

{'meteo': {'valeur': 'soleil',
  'gauche': {'temperature': {'valeur': 'froid',
    'gauche': 'oui',
    'droite': 'non'}},
  'droite': {'vent': {'valeur': 'faible',
    'gauche': 'oui',
    'droite': {'meteo': {'valeur': 'pluie',
      'gauche': 'non',
      'droite': 'oui'}}}}}}

In [36]:
def predire_cart(arbre, observation):
    """
    Prédiction avec un arbre CART.
    """
    
    # Si feuille
    if not isinstance(arbre, dict):
        return arbre
    
    # Variable du noeud
    feature = next(iter(arbre))
    
    # Valeur du split
    valeur_split = arbre[feature]["valeur"]
    
    # Décision gauche/droite
    if observation[feature] == valeur_split:
        
        return predire_cart(
            arbre[feature]["gauche"],
            observation
        )
    
    else:
        
        return predire_cart(
            arbre[feature]["droite"],
            observation
        )

In [37]:
predictions_cart = []

for i in range(len(data_id3)):
    
    observation = data_id3.iloc[i]
    
    prediction = predire_cart(
        arbre_cart,
        observation
    )
    
    predictions_cart.append(prediction)

# Ajouter les prédictions au dataset
data_id3["prediction_cart"] = predictions_cart

data_id3

,meteo,temperature,vent,jouer,prediction_id3,prediction_cart
0,soleil,chaud,faible,non,non,non
1,soleil,chaud,fort,non,non,non
2,nuage,chaud,faible,oui,oui,oui
3,pluie,doux,faible,oui,oui,oui
4,pluie,froid,faible,oui,oui,oui
5,pluie,froid,fort,non,non,non
6,nuage,froid,fort,oui,oui,oui
7,soleil,doux,faible,non,non,non
8,soleil,froid,faible,oui,oui,oui
9,pluie,doux,faible,oui,oui,oui


Différence finale :

ID3	 : Entropie + Gain et Multi-branche

C4.5 :	Gain Ratio et Multi-branche

CART :	Gini et Binaire